# 🗣️ Vaani — Phase 1 Setup Notebook
## Colab / Kaggle T4 GPU Environment

**Purpose:** Prove each of the 4 core models works in isolation on your own data.

**Run cells in order. Do NOT skip cells.**

---
### Before you start:
1. **Runtime → Change runtime type → T4 GPU** (Colab) or enable GPU accelerator (Kaggle)
2. Have a 30-90 second WAV/MP4 clip of yourself speaking ready to upload
3. This notebook creates a `vaani/` directory in your working directory

---
### ⚠️ Dependency Warning
Wav2Lip has strict dependency requirements. **Do not upgrade numpy, librosa, or face-alignment** past the pinned versions without re-testing. If you hit errors, check `notes.md` first.

## STEP 1 — GPU Detection
Confirm T4 GPU is available before installing anything.

In [ ]:
import subprocess
import sys

# --- GPU Detection ---
print('=== GPU CHECK ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    # Parse GPU name and memory
    for line in result.stdout.split('\n'):
        if 'Tesla' in line or 'T4' in line or 'A100' in line or 'P100' in line or 'V100' in line:
            print(f'  GPU detected: {line.strip()}')
    print('  ✅ GPU is available')
else:
    print('  ❌ No GPU detected!')
    print('  Go to: Runtime → Change runtime type → T4 GPU')
    sys.exit('GPU required. Aborting.')

# --- PyTorch CUDA check ---
try:
    import torch
    print(f'\n  PyTorch version: {torch.__version__}')
    print(f'  CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'  GPU name: {torch.cuda.get_device_name(0)}')
        total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'  Total VRAM: {total_mem:.1f} GB')
        if total_mem < 12:
            print(f'  ⚠️  WARNING: <12 GB VRAM detected. Some models may OOM.')
        else:
            print(f'  ✅ VRAM sufficient for all Phase 1 models')
except ImportError:
    print('  PyTorch not installed yet — will be installed in Step 2')

print('\n=== GPU CHECK COMPLETE ===')

## STEP 2 — System Dependencies
Install ffmpeg and build tools. **Run once per Colab session.**

In [ ]:
print('Installing system dependencies...')
!apt-get update -qq
!apt-get install -y -qq ffmpeg cmake build-essential libsndfile1
print('✅ System dependencies installed')

# Verify ffmpeg
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    ffmpeg_version = result.stdout.split('\n')[0]
    print(f'  ffmpeg: {ffmpeg_version}')
else:
    print('  ❌ ffmpeg not found — Whisper and pydub will fail!')

## STEP 3 — Install PyTorch (Pinned Version)

⚠️ **CRITICAL: Install PyTorch BEFORE all other packages.**
Installing TTS or other packages first will pull in a newer PyTorch that breaks Wav2Lip.

Target: `torch==1.12.1` with CUDA 11.3 (works on Colab T4)

In [ ]:
print('Installing pinned PyTorch 1.12.1 + CUDA 11.3...')
print('This takes 2-3 minutes on first run...')

!pip install -q \
    torch==1.12.1+cu113 \
    torchvision==0.13.1+cu113 \
    torchaudio==0.12.1+cu113 \
    --extra-index-url https://download.pytorch.org/whl/cu113

# Verify
import importlib, sys
# Reload in case a different version was previously imported
for mod in ['torch', 'torchvision']:
    if mod in sys.modules:
        del sys.modules[mod]

import torch
print(f'\n✅ torch=={torch.__version__}')
print(f'   CUDA available: {torch.cuda.is_available()}')
assert '1.12' in torch.__version__, f'Expected 1.12.x, got {torch.__version__}. Check pip output above for errors.'

## STEP 4 — Install Core Python Packages

Installing packages in dependency order. NumPy is pinned to <1.24 for Wav2Lip compatibility.

In [ ]:
print('Installing core packages...')

# NumPy FIRST — pinned for Wav2Lip (np.bool removed in 1.24+)
!pip install -q numpy==1.23.5
print('  ✅ numpy==1.23.5')

# Audio stack
!pip install -q librosa==0.8.1 soundfile==0.10.3 audioread==2.1.9 pydub==0.25.1 ffmpeg-python==0.2.0
print('  ✅ audio stack (librosa==0.8.1, soundfile, pydub)')

# OpenCV
!pip install -q opencv-python==4.5.5.64 opencv-contrib-python==4.5.5.64
print('  ✅ opencv-python==4.5.5.64')

# Pillow (pinned for Wav2Lip compat)
!pip install -q Pillow==9.5.0
print('  ✅ Pillow==9.5.0')

# Utils
!pip install -q tqdm requests python-dotenv
print('  ✅ utils (tqdm, requests, python-dotenv)')

# Verify numpy didn't get upgraded
import numpy as np
assert np.__version__ == '1.23.5', f'NumPy version mismatch: {np.__version__}. Something upgraded it!'
print(f'\n✅ numpy=={np.__version__} — version confirmed correct')

## STEP 5 — Install Whisper (ASR)

In [ ]:
!pip install -q openai-whisper==20231117

import whisper
print(f'✅ openai-whisper installed')
print(f'   Available models: {whisper.available_models()}')

## STEP 6 — Install IndicTrans2 (Translation)

IndicTrans2 installs from source. Takes ~3 minutes on first run.

In [ ]:
import os

# Clone IndicTrans2 repository (skip if already cloned)
if not os.path.isdir('IndicTrans2'):
    print('Cloning IndicTrans2...')
    !git clone -q https://github.com/AI4Bharat/IndicTrans2.git
    print('  ✅ Cloned')
else:
    print('  ✅ IndicTrans2 already cloned')

# Install from source
print('Installing IndicTrans2 and dependencies...')
!pip install -q -e IndicTrans2/ 2>&1 | tail -5

# Install additional IndicTrans2 dependencies
!pip install -q transformers==4.35.2 sentencepiece==0.1.99 sacremoses==0.0.53
!pip install -q indic-nlp-library==0.91 mosestokenizer==1.2.1 2>&1 | tail -3

# Verify
try:
    from IndicTransToolkit import IndicProcessor
    print('\n✅ IndicTrans2 installed and importable')
    print('   IndicProcessor importable: ✅')
except ImportError as e:
    print(f'\n❌ IndicTransToolkit import failed: {e}')
    print('   Try: !pip install -e IndicTrans2/huggingface_interface/')
    print('   See notes.md for troubleshooting')

## STEP 7 — Install XTTS-v2 (Voice Cloning)

Coqui TTS installs XTTS-v2. Takes ~2 minutes. Model weights (~2 GB) download on first use.

In [ ]:
print('Installing Coqui TTS (XTTS-v2)...')
!pip install -q TTS==0.22.0 2>&1 | tail -5

# Verify — don't import TTS yet (it may try to download model weights)
result = subprocess.run([sys.executable, '-c', 'from TTS.api import TTS; print("TTS importable")'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ {result.stdout.strip()}')
else:
    print(f'❌ TTS import failed: {result.stderr[-500:]}')

# Check that numpy is still the correct version (TTS can sometimes upgrade it)
import numpy as np
if np.__version__ != '1.23.5':
    print(f'\n⚠️  numpy was upgraded to {np.__version__} by TTS install!')
    print('   Downgrading back to 1.23.5...')
    !pip install -q numpy==1.23.5
    print('   ✅ numpy downgraded')
else:
    print(f'   numpy=={np.__version__} — version intact ✅')

## STEP 8 — Install Wav2Lip (Lip Sync)

⚠️ **Highest-risk step.** Follow carefully.

Wav2Lip requires:
1. Cloning the repo
2. Downloading the pretrained checkpoint manually
3. Installing face-alignment at the exact pinned version

In [ ]:
# --- Clone Wav2Lip ---
if not os.path.isdir('Wav2Lip'):
    print('Cloning Wav2Lip...')
    !git clone -q https://github.com/Rudrabha/Wav2Lip.git
    print('  ✅ Cloned')
else:
    print('  ✅ Wav2Lip already cloned')

# --- Install face-alignment at EXACT pinned version ---
print('\nInstalling face-alignment==1.3.5 (EXACT version required)...')
!pip install -q face-alignment==1.3.5

# Verify face-alignment version
import face_alignment
print(f'  face_alignment version: {face_alignment.__version__}')
if face_alignment.__version__ != '1.3.5':
    print(f'  ⚠️  Expected 1.3.5, got {face_alignment.__version__}')
    print(f'  This may cause Wav2Lip to fail. Check notes.md.')
else:
    print(f'  ✅ face_alignment==1.3.5 confirmed')

print('\n✅ Wav2Lip cloned and face detection installed')
print('   NEXT STEP: Download wav2lip_gan.pth checkpoint (see cell below)')

In [ ]:
# --- Download Wav2Lip GAN checkpoint ---
# The checkpoint is ~400 MB. Download from the official release.

import os

checkpoint_dir = 'Wav2Lip/checkpoints'
checkpoint_path = f'{checkpoint_dir}/wav2lip_gan.pth'

os.makedirs(checkpoint_dir, exist_ok=True)

if os.path.isfile(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / 1e6
    print(f'✅ Checkpoint already exists: {checkpoint_path} ({size_mb:.0f} MB)')
else:
    print('Downloading wav2lip_gan.pth...')
    # Official download link from Rudrabha's Google Drive (via gdown)
    !pip install -q gdown
    # File ID from: https://github.com/Rudrabha/Wav2Lip#getting-the-weights
    !gdown --id 1P4ifX9DE2EJ7RdoEm0qDMpECMax7RQHM -O {checkpoint_path}

    if os.path.isfile(checkpoint_path):
        size_mb = os.path.getsize(checkpoint_path) / 1e6
        print(f'✅ Downloaded: {checkpoint_path} ({size_mb:.0f} MB)')
    else:
        print('❌ Download failed. Manual download required:')
        print('   1. Go to: https://github.com/Rudrabha/Wav2Lip/releases')
        print('   2. Download wav2lip_gan.pth')
        print(f'   3. Upload to: {checkpoint_path}')

# Set environment variable for Vaani's lipsync module
os.environ['WAV2LIP_REPO_PATH'] = os.path.abspath('Wav2Lip')
os.environ['WAV2LIP_CHECKPOINT_PATH'] = os.path.abspath(checkpoint_path)
print(f'\n   WAV2LIP_REPO_PATH={os.environ["WAV2LIP_REPO_PATH"]}')
print(f'   WAV2LIP_CHECKPOINT_PATH={os.environ["WAV2LIP_CHECKPOINT_PATH"]}')

## STEP 9 — Version Audit

Print ALL critical package versions. Copy this output to `notes.md`.

In [ ]:
import importlib
import sys

CRITICAL_PACKAGES = [
    ('torch', 'torch'),
    ('torchvision', 'torchvision'),
    ('numpy', 'numpy'),
    ('opencv-python', 'cv2'),
    ('librosa', 'librosa'),
    ('face-alignment', 'face_alignment'),
    ('soundfile', 'soundfile'),
    ('pydub', 'pydub'),
    ('transformers', 'transformers'),
    ('sentencepiece', 'sentencepiece'),
    ('TTS', 'TTS'),
    ('Pillow', 'PIL'),
    ('tqdm', 'tqdm'),
]

print('=' * 60)
print('DEPENDENCY VERSION AUDIT')
print('=' * 60)
print(f'Python: {sys.version}')
print()

all_ok = True
for pkg_name, import_name in CRITICAL_PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f'  {pkg_name:<25} {version}')
    except ImportError:
        print(f'  {pkg_name:<25} ❌ NOT INSTALLED')
        all_ok = False

print()
print('=' * 60)
if all_ok:
    print('✅ All packages installed')
else:
    print('❌ Some packages missing — check cells above for errors')
print('=' * 60)
print('\nCopy the output above to notes.md → Phase 1 Log')

## STEP 10 — Clone Vaani Repository

Clone the Vaani project and install it in editable mode so modules are importable.

In [ ]:
if not os.path.isdir('vaani'):
    print('Cloning Vaani repository...')
    !git clone -q https://github.com/SAK_SHI14/vaani.git
    print('  ✅ Cloned')
else:
    print('  ✅ vaani/ already present')

%cd vaani

# Install Vaani itself in editable mode
!pip install -q -e . 2>&1 | tail -3
print('✅ Vaani installed in editable mode')

# Verify import
import vaani as _mimi
print(f'   vaani version: {_vaani.__version__}')

## STEP 11 — Upload Your Source Clips

Upload your self-recorded clips. You need:
- **1 audio clip** (WAV, MP3) for ASR and voice cloning tests
- **1 video clip** (MP4) with your face visible for the lip-sync test
- The audio clip should be ≥10 seconds for XTTS-v2 voice cloning

In [ ]:
import os
os.makedirs('samples', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Upload your files via Colab's file upload widget
try:
    from google.colab import files
    print('Upload your audio clip (WAV/MP3):')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        save_path = f'samples/{fname}'
        with open(save_path, 'wb') as f:
            f.write(data)
        print(f'  Saved: {save_path} ({len(data)/1e6:.1f} MB)')
except ImportError:
    # On Kaggle — use dataset files or manual path
    print('Not running in Colab. Place your files in samples/ manually.')

# Show what's in samples/
print('\nFiles in samples/:')
for f in os.listdir('samples'):
    size = os.path.getsize(f'samples/{f}') / 1e6
    print(f'  {f} ({size:.1f} MB)')

## STEP 12 — Test 1: Whisper ASR

Transcribe your uploaded audio clip. Review the output for accuracy.

In [ ]:
import os

# ─── Set this to your uploaded audio file ───
AUDIO_FILE = 'samples/your_clip.wav'  # ← UPDATE THIS
# ────────────────────────────────────────────

if not os.path.isfile(AUDIO_FILE):
    print(f'❌ File not found: {AUDIO_FILE}')
    print('   Update AUDIO_FILE to the path of your uploaded clip')
else:
    from vaani.asr.transcribe import transcribe
    print(f'Transcribing: {AUDIO_FILE}')
    print('Using Whisper medium model (~1.5 GB VRAM)...')

    transcript = transcribe(AUDIO_FILE, model_size='medium', language='en')

    print('\n' + '='*60)
    print('TRANSCRIPT:')
    print('='*60)
    print(transcript)
    print('='*60)
    print('\n✅ ASR complete. Review the transcript for accuracy.')
    print('   Save this result — you will use it as input to the translation step.')

## STEP 13 — Test 2: IndicTrans2 Translation

Translate the transcript from Step 12 into Hindi.

⚠️ First run will download the 1B model (~4 GB). Takes ~5 minutes on T4.

In [ ]:
# ─── Paste your transcript from Step 12 here ───
ENGLISH_TEXT = transcript  # uses the transcript from the cell above; or hardcode a sentence
# ENGLISH_TEXT = "Hello, my name is Sakshi and I am excited to show you this project."
# ───────────────────────────────────────────────

from vaani.translation.translate import translate

print('Translating English → Hindi...')
print(f'Input ({len(ENGLISH_TEXT)} chars): "{ENGLISH_TEXT[:100]}..."')
print('Loading IndicTrans2-1B (first run: ~5 min download)...')

hindi_text = translate(ENGLISH_TEXT, source_lang='eng_Latn', target_lang='hin_Deva')

print('\n' + '='*60)
print('HINDI TRANSLATION:')
print('='*60)
print(hindi_text)
print('='*60)
print('\n✅ Translation complete.')
print('   Review: does this look like valid Devanagari Hindi?')
print('   (You can paste it into Google Translate to check)')

## STEP 14 — Test 3: XTTS-v2 Voice Cloning

Synthesize an English sentence in your cloned voice. We test English first to isolate voice cloning from translation.

⚠️ First run downloads XTTS-v2 weights (~2 GB). Takes ~3 minutes.

In [ ]:
import os
from IPython.display import Audio, display

# ─── Settings ───
REFERENCE_AUDIO = 'samples/your_clip.wav'  # ← UPDATE: must be ≥6s of your voice
TEST_TEXT_EN = "This is a test sentence to verify voice cloning is working correctly."
TEST_TEXT_HI = "यह एक परीक्षण वाक्य है जो यह सत्यापित करने के लिए है कि आवाज़ क्लोनिंग सही ढंग से काम कर रही है।"
OUTPUT_EN = 'outputs/cloned_english_test.wav'
OUTPUT_HI = 'outputs/cloned_hindi_test.wav'
# ─────────────────

from vaani.voice_cloning.clone_voice import clone_voice

# --- English test (isolates voice cloning from translation) ---
print('Test A: Cloning voice (English)...')
print('Loading XTTS-v2 (first run: ~3 min download)...')

out_en = clone_voice(
    reference_audio_path=REFERENCE_AUDIO,
    text=TEST_TEXT_EN,
    language='en',
    output_path=OUTPUT_EN,
)

print(f'\n✅ English cloned voice saved: {out_en}')
print('   Listen and confirm it sounds like your voice:')
display(Audio(out_en))

# --- Hindi test (voice cloning + translation) ---
print('\nTest B: Cloning voice (Hindi)...')
out_hi = clone_voice(
    reference_audio_path=REFERENCE_AUDIO,
    text=TEST_TEXT_HI,
    language='hi',
    output_path=OUTPUT_HI,
)

print(f'\n✅ Hindi cloned voice saved: {out_hi}')
print('   Listen — should sound like your voice speaking Hindi:')
display(Audio(out_hi))
print('   Expected: voice identity recognizable, pronunciation may sound slightly accented')
print('   Known limitation: Hindi quality lower than English (documented in ADR-002)')

## STEP 15 — Test 4: Wav2Lip Lip Sync

Re-sync a video to its OWN original audio first (baseline sanity check).

⚠️ This is the highest-risk step. If it fails, read the error message carefully — it will contain a specific diagnosis.

In [ ]:
import os
from IPython.display import Video, display

# ─── Settings ───
SOURCE_VIDEO = 'samples/your_video.mp4'  # ← UPDATE: must show your face clearly
# For this test, we use the video's OWN audio (extracted below)
EXTRACTED_AUDIO = 'outputs/extracted_audio_test.wav'
OUTPUT_VIDEO = 'outputs/lipsync_baseline_test.mp4'
# ─────────────────

# Step 1: Extract audio from video (proves ffmpeg is working)
print('Extracting audio from source video...')
!ffmpeg -i {SOURCE_VIDEO} -vn -acodec pcm_s16le -ar 16000 -ac 1 {EXTRACTED_AUDIO} -y -loglevel error

if os.path.isfile(EXTRACTED_AUDIO):
    size = os.path.getsize(EXTRACTED_AUDIO) / 1e6
    print(f'  ✅ Audio extracted: {EXTRACTED_AUDIO} ({size:.1f} MB)')
else:
    print(f'  ❌ Audio extraction failed. Check that {SOURCE_VIDEO} exists and is a valid video.')

# Step 2: Run Wav2Lip
print('\nRunning Wav2Lip lip-sync...')
print('   (video → same video re-synced to its own audio — baseline test)')

from vaani.lipsync.lipsync import lipsync

out_video = lipsync(
    video_path=SOURCE_VIDEO,
    audio_path=EXTRACTED_AUDIO,
    output_path=OUTPUT_VIDEO,
)

print(f'\n✅ Lip-synced video saved: {out_video}')
print('   Watch the output and compare to the original:')
print('   Expected: lip movements should look approximately natural')
print('   Known limitation: artifacts possible on fast movement or profile angles')
display(Video(out_video, width=640))

## STEP 16 — Run Unit Tests

All unit tests use mocked models — they should pass without GPU.

In [ ]:
!pip install -q pytest pytest-timeout

print('Running unit tests (models are mocked — no GPU needed)...')
!pytest asr/tests/ translation/tests/ voice_cloning/tests/ lipsync/tests/ -v --tb=short 2>&1

## STEP 17 — Phase 1 Complete Checklist

Run this cell to auto-check the Phase 1 deliverables.

In [ ]:
import os

checks = [
    ('GPU available', lambda: __import__('torch').cuda.is_available()),
    ('ffmpeg installed', lambda: os.system('ffmpeg -version > /dev/null 2>&1') == 0),
    ('Whisper importable', lambda: bool(__import__('whisper'))),
    ('IndicTransToolkit importable', lambda: bool(__import__('IndicTransToolkit'))),
    ('TTS importable', lambda: bool(__import__('TTS'))),
    ('face_alignment importable', lambda: bool(__import__('face_alignment'))),
    ('Wav2Lip inference.py exists', lambda: os.path.isfile('Wav2Lip/inference.py')),
    ('Wav2Lip checkpoint exists', lambda: os.path.isfile('Wav2Lip/checkpoints/wav2lip_gan.pth')),
    ('numpy==1.23.5', lambda: __import__('numpy').__version__ == '1.23.5'),
    ('ADR-001 written', lambda: os.path.isfile('docs/adrs/ADR-001-translation-model.md')),
    ('ADR-002 written', lambda: os.path.isfile('docs/adrs/ADR-002-voice-cloning-model.md')),
    ('notes.md exists', lambda: os.path.isfile('notes.md')),
    ('outputs/cloned_english_test.wav exists', lambda: os.path.isfile('outputs/cloned_english_test.wav')),
    ('outputs/lipsync_baseline_test.mp4 exists', lambda: os.path.isfile('outputs/lipsync_baseline_test.mp4')),
]

print('=' * 60)
print('PHASE 1 COMPLETION CHECKLIST')
print('=' * 60)

passed = 0
failed = 0
for name, check_fn in checks:
    try:
        result = check_fn()
        status = '✅' if result else '❌'
        if result:
            passed += 1
        else:
            failed += 1
    except Exception as e:
        status = f'❌ ({type(e).__name__})'
        failed += 1
    print(f'  {status}  {name}')

print('=' * 60)
print(f'  Passed: {passed}/{len(checks)}')
if failed == 0:
    print('  🎉 Phase 1 complete! Ready for Phase 2 review.')
else:
    print(f'  ❌ {failed} checks failing — resolve before Phase 2.')
print('=' * 60)